# Run NMME EMOV Processing

This notebook drives `scripts/run_process_modes_of_variability.py` for NMME gridded SST modes of variability.

It mirrors the model selection and archive checks from `0_run_nmme_sst_index.ipynb`, but processes full gridded fields for projected EOF mode diagnostics. NMME SST modes use `sst`; pressure modes use NMME `prmsl`.

Primary outputs are written below:

`/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/nmme/modes_variability/`


In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
import xarray as xr

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPT_PATH = REPO_ROOT / "scripts" / "run_process_modes_of_variability.py"
print(f"Repository root: {REPO_ROOT}")
print(f"Python         : {sys.executable}")
print(f"Script         : {SCRIPT_PATH}")


## Configuration

Leave `MODELS = []` to use `MODEL_SET`, or keep the explicit list below to match the NMME SST-index preprocessing notebook. The notebook automatically uses only models that have the required raw variable: `sst` for SST modes and `prmsl` for pressure modes.


In [ ]:
NMME_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member")
OUTDIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag")
MODEL_SET = "all"  # Used only when MODELS is empty: "all" or "yeager-f03".

MODES = ["NAM", "NAO", "SAM", "PSA1", "PSA2", "PNA", "NPO", "EA", "SCA", "PDO", "NPGO", "AMO"]
INIT_MONTHS = [2, 5, 8, 11]

# NMME SST files typically provide 12 monthly leads.
START_YEAR = "common"  # Use "common" for selected-model overlap, or set an integer year.
END_YEAR = "common"    # Use "common" for selected-model overlap, or set an integer year.
CLIM_START = 1982
CLIM_END = 2010
MONTHLY_NLEAD = 12

OBS_START_YEAR = 1950
OBS_END_YEAR = 2020
EOF_REFERENCE_START_YEAR = 1950
EOF_REFERENCE_END_YEAR = 2020

TARGET_DLAT = 2.5
TARGET_DLON = 2.5
DASK_WORKERS = 8
NMME_CHUNKS = ""  # Empty uses archive/native chunks; override with e.g. "S:12,L:-1,Y:181,X:360".
FORCE = False
DRY_RUN = False
RUN_DRY_RUN_FIRST = True

MODELS = [
    "CanSIPS-IC3",
    "GFDL-CM2p1",
    "NASA-GMAO-062012",
    "CanSIPS-IC4",
    "GFDL-CM2p1-aer04",
    "NCAR-CESM1",
    "CanSIPSv2",
    "GFDL-CM2p5-FLOR-A06",
    "NCEP-CFSv1",
    "CMC1-CanCM3",
    "GFDL-CM2p5-FLOR-B01",
    "NCEP-CFSv2",
    "CMC2-CanCM4",
    "GFDL-SPEAR",
    "GEM-NEMO",
    "NASA-GMAO",
    "CanCM4i",
    "NASA-GEOSS2S",
    "COLA-RSMAS-CCSM3",
    "IRI-ECHAM4p5-AnomalyCoupled",
    "COLA-RSMAS-CCSM4",
    "IRI-ECHAM4p5-DirectCoupled",
    "COLA-RSMAS-CESM1"
]

YEAGER_F03_MODELS = [
    "CanCM4i",
    "COLA-RSMAS-CCSM4",
    "GEM-NEMO",
    "GFDL-CM2p5-FLOR-B01",
    "NASA-GMAO",
    "NCEP-CFSv2",
    "CanSIPSv2",
    "GFDL-SPEAR",
]


## Validate Inputs

This cell follows the NMME SST-index notebook's archive checks, including explicit model selection and rough initialization-year coverage.


In [ ]:
missing = []
for label, candidate in {
    "script": SCRIPT_PATH,
    "NMME root": NMME_ROOT,
}.items():
    if not candidate.exists():
        missing.append(f"{label}: {candidate}")

if MODEL_SET not in {"all", "yeager-f03"}:
    missing.append(f"MODEL_SET must be 'all' or 'yeager-f03', got {MODEL_SET!r}")

if missing:
    raise FileNotFoundError("Missing or invalid configuration:
" + "
".join(missing))

available_models = sorted(
    path.name for path in NMME_ROOT.iterdir()
    if path.is_dir() and path.name != "logs" and (path / "sst").is_dir()
)

EXPLICIT_MODELS = list(
    dict.fromkeys(str(model).strip() for model in MODELS if str(model).strip())
)
selection_errors = []
if EXPLICIT_MODELS:
    unavailable_models = sorted(set(EXPLICIT_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODELS contains names that are not available under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = EXPLICIT_MODELS
    model_selection = "explicit MODELS list"
elif MODEL_SET == "all":
    SELECTED_MODELS = available_models
    model_selection = "MODEL_SET='all'"
else:
    unavailable_models = sorted(set(YEAGER_F03_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODEL_SET='yeager-f03' includes unavailable models under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = YEAGER_F03_MODELS
    model_selection = "MODEL_SET='yeager-f03'"

if selection_errors:
    raise ValueError("Invalid model selection:
" + "
".join(selection_errors))

s_chunk_pattern = re.compile(r"_S(\d+)-(\d+)\.nc$")

def _chunk_range(path):
    match = s_chunk_pattern.search(path.name)
    return (int(match.group(1)), int(match.group(2))) if match else None

def _decode_s_edge(path, first=True):
    with xr.open_dataset(path, decode_times=False) as ds:
        s_ds = ds[["S"]].copy()
    if s_ds["S"].attrs.get("calendar") == "360":
        s_ds["S"].attrs["calendar"] = "360_day"
    decoded = xr.decode_cf(s_ds, decode_times=True)["S"]
    return decoded.values[0 if first else -1]

def _model_s_years(model):
    files = []
    for path in (NMME_ROOT / model / "sst").glob("M*/*.nc"):
        chunk_range = _chunk_range(path)
        if chunk_range is not None:
            files.append((*chunk_range, path))
    if not files:
        raise FileNotFoundError(f"No SST chunks found for {model}")
    first_file = min(files, key=lambda item: (item[0], item[2]))[2]
    last_file = max(files, key=lambda item: (item[1], item[2]))[2]
    first = _decode_s_edge(first_file, first=True)
    last = _decode_s_edge(last_file, first=False)
    return int(first.year), int(last.year)

model_year_ranges = {model: _model_s_years(model) for model in SELECTED_MODELS}
coverage = pd.DataFrame(
    [
        {"model": model, "first_year": years[0], "last_year": years[1]}
        for model, years in model_year_ranges.items()
    ]
).sort_values("model")

common_start_year = max(start for start, _ in model_year_ranges.values())
common_end_year = min(end for _, end in model_year_ranges.values())
START_YEAR_RESOLVED = common_start_year if START_YEAR == "common" else int(START_YEAR)
END_YEAR_RESOLVED = common_end_year if END_YEAR == "common" else int(END_YEAR)

if START_YEAR_RESOLVED > END_YEAR_RESOLVED:
    raise ValueError(
        f"Resolved START_YEAR ({START_YEAR_RESOLVED}) must be <= END_YEAR ({END_YEAR_RESOLVED})."
    )
if not (START_YEAR_RESOLVED <= CLIM_START <= CLIM_END <= END_YEAR_RESOLVED):
    raise ValueError(
        "CLIM_START/CLIM_END must lie within the resolved NMME processing years: "
        f"{START_YEAR_RESOLVED}-{END_YEAR_RESOLVED}"
    )

year_errors = []
for model, (first_year, last_year) in model_year_ranges.items():
    if first_year > START_YEAR_RESOLVED or last_year < END_YEAR_RESOLVED:
        year_errors.append(
            f"{model}: available {first_year}-{last_year}, requested "
            f"{START_YEAR_RESOLVED}-{END_YEAR_RESOLVED}"
        )
if year_errors:
    print("Some selected models do not fully cover the resolved years:")
    for item in year_errors:
        print(" ", item)

print(f"Selected {len(SELECTED_MODELS)} NMME models via {model_selection}")
print(f"Common selected-model years: {common_start_year}-{common_end_year}")
print(f"Resolved years: {START_YEAR_RESOLVED}-{END_YEAR_RESOLVED}; climatology: {CLIM_START}-{CLIM_END}")
display(coverage)


## Build Command


In [ ]:
def build_command(dry_run=False):
    command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--outdir", str(OUTDIR),
        "--modes", *MODES,
        "--sources", "obs", "nmme",
        "--init-months", *(str(month) for month in INIT_MONTHS),
        "--start-year", str(START_YEAR_RESOLVED),
        "--end-year", str(END_YEAR_RESOLVED),
        "--clim-start", str(CLIM_START),
        "--clim-end", str(CLIM_END),
        "--obs-start-year", str(OBS_START_YEAR),
        "--obs-end-year", str(OBS_END_YEAR),
        "--eof-reference-start-year", str(EOF_REFERENCE_START_YEAR),
        "--eof-reference-end-year", str(EOF_REFERENCE_END_YEAR),
        "--monthly-nlead", str(MONTHLY_NLEAD),
        "--target-dlat", str(TARGET_DLAT),
        "--target-dlon", str(TARGET_DLON),
        "--dask-workers", str(DASK_WORKERS),
        "--nmme-root", str(NMME_ROOT),
        "--nmme-models", *SELECTED_MODELS,
        "--nmme-field", "auto",
        "--nmme-chunks", NMME_CHUNKS,
        "--ts-obs-product", "HadISST2",
        "--ts-obs-var", "sst",
        "--eof-strategy", "fixed_obs_projection",
        "--eof-bootstrap-iterations", "0",
        "--merge-manifest",
    ]
    if FORCE:
        command.append("--force")
    if dry_run or DRY_RUN:
        command.append("--dry-run")
    return command

cmd = build_command(dry_run=False)
print("Command:")
print(" ".join(map(str, cmd)))


## Dry Run


In [ ]:
if RUN_DRY_RUN_FIRST:
    dry_cmd = build_command(dry_run=True)
    print("Dry-run command:")
    print(" ".join(map(str, dry_cmd)))
    dry_run = subprocess.run(
        dry_cmd,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    print(dry_run.stdout)
    if dry_run.stderr:
        print(dry_run.stderr)
else:
    print("Skipping dry run because RUN_DRY_RUN_FIRST=False.")


## Run Processing


In [ ]:
env = os.environ.copy()
conda_prefix = Path(sys.prefix)
for env_name, relpath in {
    "GDAL_DATA": "share/gdal",
    "PROJ_LIB": "share/proj",
    "PROJ_DATA": "share/proj",
}.items():
    candidate = conda_prefix / relpath
    if candidate.exists():
        env[env_name] = str(candidate)

def run_processing():
    command = build_command(dry_run=DRY_RUN)
    print("Running command:")
    print(" ".join(map(str, command)))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        check=True,
    )
    return completed

completed = run_processing()
completed.returncode


## Product Summary


In [ ]:
manifest_path = OUTDIR / "modes_manifest.json"
if not manifest_path.exists():
    print("No manifest found yet:", manifest_path)
else:
    manifest = json.loads(manifest_path.read_text())
    rows = []
    for product, details in manifest.get("products", {}).items():
        mode, source = product.split(":", 1)
        if mode not in MODES or not (source == "reference" or source.startswith("nmme_init")):
            continue
        index_path = Path(details["index"])
        field_path = Path(details["field"])
        rows.append(
            {
                "product": product,
                "status": details.get("status", ""),
                "index_exists": index_path.exists(),
                "field_exists": field_path.exists(),
                "index_MB": round(index_path.stat().st_size / 1e6, 2) if index_path.exists() else None,
                "field_GB": round(field_path.stat().st_size / 1e9, 2) if field_path.exists() else None,
            }
        )
    summary = pd.DataFrame(rows).sort_values("product").reset_index(drop=True)
    display(summary)
    print("Manifest:", manifest_path)
